Usando el modelo XGBoost con sus mejores parámetros

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split, GridSearchCV

from xgboost import XGBClassifier
from sklearn.linear_model import LogisticRegression

In [ ]:
abandono = pd.read_csv('./Abandono/abandono.csv',index_col=0)
abandono

In [ ]:
grades_per_exam = pd.read_csv('./Grades/grades_per_exam.csv',index_col=0)
grades_per_exam

In [ ]:
def abandono_pipeline(w, indicadores):
  theta = pd.read_csv('./Modelos/Cummulative/Configuración al menos un evaluable/theta_one_week{}.csv'.format(w),index_col=0)
  w_ind = indicadores.loc[theta.index]
  w_ind['theta'] = theta['conf1']
  w_ind['passed'] = abandono['curso_superado']
  w_ind['dropped_out'] = abandono['abandono']

  if 'CFA' in w_ind.columns:
    w_ind['CFA'] = w_ind['CFA'].replace('0 problems attempted',0)
  if 'avg_grade' in w_ind.columns:
    w_ind['avg_grade'] = w_ind['avg_grade'].replace('0 problems attempted',0)
  if 'avg_attempts' in w_ind.columns:
    w_ind['avg_attempts'] = w_ind['avg_attempts'].replace('0 problems attempted',0)

  w_ind['dropped_out'] = w_ind['dropped_out'].replace('No',0).replace('Sí',1)
  w_ind['passed'] = w_ind['passed'].replace('No',0).replace('Sí',1)

  X = w_ind.drop(['dropped_out','passed'],axis=1)
  y = w_ind['dropped_out']

  X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)

  pipe_xgb = Pipeline([('scl', StandardScaler()),
                      ('XGB', XGBClassifier(random_state=42))])

  xgb_param_grid = [{'XGB__learning_rate': [0.1],
                      'XGB__max_depth': [5],
                      'XGB__min_child_weight': [1.0],
                      'XGB__subsample': [1.0],
                      'XGB__n_estimators': [75]}]

  xgb_grid_search = GridSearchCV(estimator=pipe_xgb,
          param_grid=xgb_param_grid,
          scoring='roc_auc',
          cv=3)
  grids = [xgb_grid_search]
  names = ['xgb']

  n = 0
  for pipe in grids:
    print(names[n])
    pipe.fit(X_train,y_train)
    n+=1

  aucs = {}
  grid_dict = {0: 'XGBoost'}
  #grid_dict = {0: 'Logistic Regression'}
  for i, model in enumerate(grids):
    aucs = model.score(X_test,y_test)
    #print(model.best_params_)

  return aucs

In [ ]:
## Variables:
# 'n_attempted', 'per_attempted', 'CFA', 'avg_grade', 'avg_attempts',
# 'n_show', 'per_videos', 'n_pauses', 'n_threads', 'n_responses',
# 'n_comments', 'n_days', 'n_videos_watched', 'n_videos_opened', 'theta'

In [ ]:
vars = ['n_attempted','per_attempted','CFA', 'avg_grade', 'avg_attempts','n_show','per_videos','n_pauses','n_threads','n_responses','n_comments','n_days','n_videos_watched','n_videos_opened','theta']

In [ ]:
for v in vars[:-1]:
  print(v)

In [ ]:
aucs = {}
for v in vars[:-1]:
  print(v)
  aucs[v] = {}
  for w in range(1,8):
    print('\nWeek', w)
    print('Importamos indicadores')
    ind =  pd.read_csv('./Modelos/Cummulative/Indicadores/indicadores_week1_{}.csv'.format(w),index_col=0)
    print('Columnas indicadores:',ind.drop([v],axis=1).columns)
    print('Calculamos auc sin variable:',v)
    auc =  abandono_pipeline(w, indicadores = ind.drop([v],axis=1))
    print('Resultados')
    aucs[v]['Week {}'.format(str(w))] = auc
    print(aucs[v]['Week {}'.format(str(w))])
  print(aucs)

In [ ]:
aucs

In [ ]:
from xgboost import XGBRegressor

In [ ]:
def nota_final_pipeline(w, indicadores):
  theta = pd.read_csv('./Modelos/Cummulative/Configuración al menos un evaluable/theta_one_week{}.csv'.format(w),index_col=0)
  w_ind = ind.loc[theta.index]
  w_ind['theta'] = theta['conf1']
  w_ind['grade'] = grades_per_exam['Course grade']
  w_ind['grade'] = w_ind['grade'].fillna(0)

  if 'CFA' in w_ind.columns:
      w_ind['CFA'] = w_ind['CFA'].replace('0 problems attempted',0)
  if 'avg_grade' in w_ind.columns:
      w_ind['avg_grade'] = w_ind['avg_grade'].replace('0 problems attempted',0)
  if 'avg_attempts' in w_ind.columns:
      w_ind['avg_attempts'] = w_ind['avg_attempts'].replace('0 problems attempted',0)

  X = w_ind.drop(['grade'],axis=1)
  y = w_ind['grade']

  X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)

  pipe_xgb = Pipeline([('scl', StandardScaler()),
                      ('XGB', XGBRegressor())])

  xgb_param_grid = [{'XGB__learning_rate': [0.1],
                      'XGB__max_depth': [5],
                      'XGB__min_child_weight': [1.0],
                      'XGB__subsample': [1.0],
                      'XGB__n_estimators': [75]}]

  xgb_grid_search = GridSearchCV(estimator=pipe_xgb,
          param_grid=xgb_param_grid,
          scoring='neg_mean_squared_error',
          cv=3)
  grids = [xgb_grid_search]
  names = ['xgb']
  n = 0
  for pipe in grids:
    print(names[n])
    pipe.fit(X_train,y_train)
    n+=1

  mses = {}
  grid_dict = {0: 'XGBoost'}
  for i, model in enumerate(grids):
    mses = -(model.score(X_test,y_test))

  return mses

In [ ]:
rmses = {}
for v in vars[:-1]:
  print(v)
  rmses[v] = {}
  for w in range(1,8):
    print('\nWeek', w)
    print('Importamos indicadores')
    ind =  pd.read_csv('./Modelos/Cummulative/Indicadores/indicadores_week1_{}.csv'.format(w),index_col=0)
    print('Columnas indicadores:',ind.drop([v],axis=1).columns)
    print('Calculamos auc sin variable:',v)
    rmse =  nota_final_pipeline(w, indicadores = ind.drop([v],axis=1))
    print('Resultados')
    rmses[v]['Week {}'.format(str(w))] = rmse
    print(rmses[v]['Week {}'.format(str(w))])
  print(rmses)

In [ ]:
rmses